# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

> **Executed against the live Hugging Face warehouse (`month=2026-03`) with a personal HF_TOKEN.** One issue was found and fixed during execution: an initial median-split toy label degenerated because daily-grain `gsc_impressions` has a median of 0 (63.3% of rows have zero impressions) — see the code comment in Section 4 for the diagnosis and the fix. All outputs below are real, rerun top to bottom.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The contract, in plain words (5 answers)

**Lane:** Freestyle — Diagnosis-First Content Triage (synthesizing Lane 2: Refresh/Content Opportunity Scoring, and Lane 4: CTR/Engagement Opportunity Scoring). See ML-02/ML-03 for the full framing.

1. **What one row means:** one row = one (`client_hash_id`, `content_hash_id`, `report_date`) record from `fact_content_daily_performance` — i.e., one content item's measured performance on one single day. This matches the table's own documented grain (`report_date + client_hash_id + content_hash_id`), so nothing needs to be invented or aggregated for this contract check.

2. **Which table(s):** `fact_content_daily_performance` (the daily fact table) as the primary source -- confirmed via a real `DESCRIBE` query to contain report_date, client_hash_id, content_hash_id, client_has_gsc/ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews/sessions/users/engaged_sessions, session-channel breakdowns (organic/direct/referral/social/paid/ai), a full AI-referral-source breakdown (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other), and scroll_events. Note: `main_intent`, `content_type`, and `trend_direction` are NOT in this table -- they live in `dim_content` (needs a separate join) or must be computed by comparing two adjacent months, not assumed present here.

3. **Time window:** a single mid-panel calendar month, `month=2026-03` (2026-03-01 through 2026-03-31), per the card's explicit warning to never develop label logic on the `_sample` table (which is only the final month, June 2026, and would leak the future outcome window into development).

4. **What I'd predict or rank (label/proxy) — TWO separate targets, kept deliberately apart:**

   - **Target A, `is_declining`:** predicts whether a page's *impressions* are trending down, built from `trend_direction` (itself comparing `impressions_last_30d` vs `impressions_prev_30d`). This is a proxy because a single 30-day-vs-30-day comparison might just be a temporary blip or noise, not a confirmed, lasting decline.
   - **Target B, `diagnosis`:** predicts *why* a flagged page is struggling — genuine_decline / likely_serp_answered / ctr_fixable / stable_or_improving — built by comparing how *impressions and clicks move together*. This is a separate proxy because even a matching pattern (e.g., impressions steady, clicks falling) is an inference about the cause, not a confirmed fact (the real query/URL is never seen, so no diagnosis can be firmly proven — see the SERP-Interception Diagnosis Framework for the full reasoning).

   **This single month cannot yet compute Target A** (it needs 60+ days of history: the current 30 days plus the prior 30) — this contract only proves the raw daily columns needed to build both targets later actually exist and are populated. See Section 4 for this named as an explicit limitation.

5. **One thing deliberately excluded:** any FlyRank product-decision column (`health_score`, `priority_score`, `action_type`) if any such column is ever visible anywhere in the warehouse — excluded because it would encode a prior decision already made by the existing system, not a raw observed signal, and using it as a feature would produce a circular result (see `SKILL.md/hunting-leakage-and-validating`). `keyword_hash_id` and `url_hash_id` are also excluded as model features — context/grouping only, per the join rules in `ml-intern-dataset-and-lane-guide.md`.

In [1]:
# Setup — run this in Colab with HF_TOKEN stored as a Secret (never pasted in a cell; this repo is public).
%pip -q install duckdb
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # Secrets panel, NOT a pasted string
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Table shapes per the Hugging Face 'Files' tab -- confirm folder-vs-single-file before running.
DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# Cheap metadata-only sanity check before touching real data.
print(con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {DAILY} WHERE month='2026-03'").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         n      min_d      max_d
0  9841378 2026-03-01 2026-03-31


In [2]:
print(con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 1").df())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `gsc_sum_position` (daily) | Feature | Raw, observed, known as soon as that day's data lands -- available at any later decision moment |
| `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` | Feature (conditional) | Observed GA4 measurements -- but only trustworthy where `ga4_data_available IS TRUE`, which is just 4.2% of this month's rows (see Section 3, Query 3) |
| `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | Feature / availability flags | Tell you whether to trust the corresponding raw measurement for that row at all |
| `sessions_organic/direct/referral/social/paid/ai`, `ai_chatgpt/perplexity/gemini/copilot/claude/meta/other` | Feature | Real, observed channel/AI-referral breakdown -- note this measures click-throughs FROM AI tools, not whether an AI Overview appeared for the query |
| `scroll_events` | Feature | Raw observed engagement signal |
| `report_date` (day of week, recency) | Feature | Purely calendar-derived -- knowable even in advance |
| `main_intent`, `content_type` | NOT in this table | Live in `dim_content` -- would require a separate join, not available directly from `fact_content_daily_performance` |
| `trend_direction`, `trend_pct` (if built later) | Label-source -- NEVER a feature | Would be computed FROM this table's impressions across two windows -- using it as an input would be circular |
| `content_hash_id`, `client_hash_id` | Context | Grouping/joining/splitting only -- never a model input |
| `health_score`, `priority_score`, `action_type` (if ever visible) | Excluded | FlyRank's own prior decision, not a raw signal -- would leak the existing system's answer back into the model |


## 3. Verify it with queries (grain, counts, missing values, windows)

Three required checks, run against `month='2026-03'` only.

In [3]:
# Query 1 -- GRAIN CHECK: one row really is one (client, content, day). Expect ZERO rows back.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {DAILY}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('Grain violations found (should be empty):')
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations found (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [4]:
# Query 2 -- ROW COUNT + DATE SPAN for this slice.
count_and_span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {DAILY}
    WHERE month = '2026-03'
""").df()
print(count_and_span)

    n_rows  n_clients  n_content_items   min_date   max_date
0  9841378         55           331437 2026-03-01 2026-03-31


In [5]:
# Query 3 -- AVAILABILITY, filtered with IS TRUE (NOT '= TRUE' and NOT 'NOT flag' -- NULL is neither).
total_rows = con.sql(f"SELECT COUNT(*) AS n FROM {DAILY} WHERE month = '2026-03'").df()['n'][0]
available_rows = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM {DAILY}
    WHERE month = '2026-03' AND ga4_data_available IS TRUE
""").df()['n'][0]

print(f"Total rows this month: {total_rows}")
print(f"Rows with GA4 data actually available (IS TRUE, not just truthy): {available_rows}")
print(f"Share surviving the availability filter: {available_rows/total_rows:.1%}")
# If this ratio looks suspiciously close to 100% or 0%, re-check for NULLs before trusting it --
# per flyrank-data: millions of rows have this flag NULL, not FALSE.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows this month: 9841378
Rows with GA4 data actually available (IS TRUE, not just truthy): 413966
Share surviving the availability filter: 4.2%


## 4. Five features (max) + the leakage trap

**Five features, each tagged with why it's knowable at the decision moment:**

1. `gsc_impressions` (daily) -- knowable because it's a closed, already-measured fact for a day that has passed.
2. `gsc_clicks` (daily) -- same reasoning; a completed day's measurement.
3. `gsc_avg_position` (daily) -- same; FlyRank/Google already recorded this ranking outcome for that day.
4. `ga4_data_available` (flag) -- knowable immediately from the row itself; tells you whether to trust GA4-derived columns for that specific day, before you use any of them.
5. `day_of_week` (derived from `report_date`) -- purely calendar arithmetic; knowable even in advance of the day itself.

**The trap, performed on purpose:** build a toy label, add one label-derived (leaky) feature, watch the score jump toward perfect, then delete it and keep the honest number.

In [6]:
# Pull a small, real feature frame for this month.
features_df = con.sql(f"""
    SELECT
        d.client_hash_id, d.content_hash_id, d.report_date,
        d.gsc_impressions, d.gsc_clicks, d.gsc_avg_position, d.ga4_data_available,
        dayofweek(d.report_date) AS day_of_week
    FROM {DAILY} d
    WHERE d.month = '2026-03'
""").df()
print(features_df.shape)
print(features_df.head())

# --- DISTRIBUTION CHECK, added after the first run revealed a degenerate median ---
print(features_df['gsc_impressions'].describe())
print("Median:", features_df['gsc_impressions'].median())
print("Share with zero impressions:", (features_df['gsc_impressions'] == 0).mean())

# --- THE TRAP ---
# FIRST ATTEMPT (kept here as the honest record of what went wrong):
# a median-split toy label crashed with "only one class: 0" -- because at this
# daily grain, the median gsc_impressions is 0 (confirmed by the check above),
# so "impressions < median" can never be true for any row. Every row got labeled
# 0, so there was nothing to learn from and nothing to split into train/test.
#
# FIXED toy label: a non-degenerate split -- did this page get ANY impressions
# that day, or none at all? This actually reflects the real, heavily zero-inflated
# shape of daily-grain data, confirmed by the distribution check above.
features_df['toy_label_low_impressions'] = (features_df['gsc_impressions'] == 0).astype(int)
print(features_df['toy_label_low_impressions'].value_counts())  # confirm BOTH classes exist now

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

honest_X = features_df[['gsc_avg_position', 'day_of_week']].fillna(0)
leaky_X  = features_df[['gsc_avg_position', 'day_of_week', 'gsc_impressions']].fillna(0)  # <-- the leak
y = features_df['toy_label_low_impressions']

for name, X in [('HONEST (no leak)', honest_X), ('LEAKY (label-derived feature included)', leaky_X)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    score = model.score(X_te, y_te)
    print(f"{name}: accuracy = {score:.3f}")

# Expect: the LEAKY version jumps toward ~1.0, because gsc_impressions == 0 is
# literally what defined the label. Delete gsc_impressions from the feature set
# (use honest_X going forward) and KEEP the honest, lower number -- that is the
# real, trustworthy result.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 8)
            client_hash_id           content_hash_id report_date  \
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6  2026-03-01   
1  client_73cda7b4e4f265ea  content_05597932fe4da067  2026-03-01   
2  client_73cda7b4e4f265ea  content_7a105f548d9c6916  2026-03-01   
3  client_73cda7b4e4f265ea  content_905aa32a0230694e  2026-03-01   
4  client_73cda7b4e4f265ea  content_a3ea9792f793ec72  2026-03-01   

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_data_available  \
0               20           0          3.350000                <NA>   
1                1           0          0.000000                <NA>   
2              125           1          4.928000                <NA>   
3                7           0          4.000000                <NA>   
4               11           0          2.272727                <NA>   

   day_of_week  
0            0  
1            0  
2            0  
3            0  
4            0  
count    9.841378e+06
mean     2.851812e+01

## Data limits (one named limitation)

**Only 4.2% of this month's rows have `ga4_data_available IS TRUE`** (413,966 of 9,841,378 rows,
confirmed directly in Section 3, Query 3). This means any GA4-derived feature (sessions, users,
engagement) can only be trusted for a small slice of the data -- the other 95.8% of rows would
need to either be excluded from GA4-dependent modeling, or handled with an explicit
"GA4 not available" category rather than being silently treated as zero engagement. This is a much
bigger practical constraint than the trend-label timing gap, and should be the first thing anyone
building on this table checks before trusting a GA4-based feature.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.